# Waves in open systems: scattering and resonances

Time-harmonic waves $e^{-i \omega t}$ in 2D (TM polarization, $E_z = u$) satisfy the Helmholtz equation

$$
-\Delta u - k^2 n^2(x) \, u = 0, \qquad k = \omega / c,
$$

with the refractive index $n$, and the Sommerfeld radiation condition at infinity.

Weak form: $\int \nabla u \cdot \nabla v - k^2 n^2 u v \, dx = \ldots$, with continuous $H^1$ elements; the interface conditions at jumps of $n$ are natural.

The unbounded domain is truncated by a **perfectly matched layer** (PML): a complex coordinate stretching $r \mapsto r + i \alpha (r - r_0)$ for $r > r_0$, under which outgoing waves decay exponentially without reflection.

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## Scattering by a disordered medium

Dielectric rods ($n = 1.5$) at random positions, illuminated by a plane wave $u_{in} = e^{i k x}$.
We solve for the scattered field $u_s = u - u_{in}$:

$$
-\Delta u_s - k^2 n^2 u_s = k^2 (n^2 - 1) \, u_{in}
$$

The source lives on the rods only, the PML sees outgoing waves only.

In [ ]:
np.random.seed(7)
centers = []
while len(centers) < 40:
    p = np.random.uniform(-1.2, 1.2, 2)
    if np.linalg.norm(p) < 1.2 and all(np.linalg.norm(p-q) > 0.25 for q in centers):
        centers.append(p)

rods = Glue([Circle(tuple(c), 0.1).Face() for c in centers])
rods.faces.name = "rod"
rods.faces.maxh = 0.04
air = Circle((0,0), 2).Face() - rods
air.faces.name = "air"
pmlregion = Circle((0,0), 2.5).Face() - Circle((0,0), 2).Face()
pmlregion.faces.name = "pml"
pmlregion.edges.Max(X).name = "outer"

mesh = Mesh(OCCGeometry(Glue([rods, air, pmlregion]), dim=2).GenerateMesh(maxh=0.1)).Curve(4)
mesh.SetPML(pml.Radial(rad=2, alpha=1j, origin=(0,0)), "pml")
Draw (mesh);

In [ ]:
k = 20
n = mesh.MaterialCF({"rod" : 1.5}, default=1)
uin = exp(1j*k*x)

fes = H1(mesh, order=5, complex=True, dirichlet="outer")
u, v = fes.TnT()
with TaskManager():
    us = Solve(grad(u)*grad(v)*dx - k**2*n**2*u*v*dx
               == k**2*(n**2-1)*uin*v*dx)
print ("ndof =", fes.ndof)

utot = mesh.MaterialCF({"pml" : 0}, default=uin + us)
Draw (utot, mesh, order=3, min=-2, max=2, animate_complex=True);

The intensity $|u|^2$: shadow, forward lobes and speckle:

In [ ]:
Draw (Norm(utot)**2, mesh, order=3, min=0, max=4);

## Resonances of a deformed microcavity

A dielectric disk $n = 3.3$ with the shape of a limaçon, $r(\varphi) = R (1 + \varepsilon \cos \varphi)$. For $\varepsilon \approx 0.43$ it has long-lived modes, which all emit into the same direction (Wiersig and Hentschel, PRL 2008).

We switch to TE polarization, $u = H_z$, with the equation $-\operatorname{div} ( n^{-2} \nabla u ) = k^2 u$. Only the bilinear forms change.

An open system has no real eigenvalues. With the PML, the resonances are the eigenvalues of a non-Hermitian (complex symmetric) problem:

$$
\int n^{-2} \, \nabla u \cdot \nabla v \, dx = k^2 \int u \, v \, dx, \qquad k \in {\mathbb C}, \quad Q = \frac{\operatorname{Re} k}{-2 \operatorname{Im} k}
$$

In [ ]:
eps, N = 0.43, 200
phis = [2*pi*i/N for i in range(N)]
pnts = [Pnt((1+eps*cos(p))*cos(p), (1+eps*cos(p))*sin(p), 0) for p in phis]
cavity = Face(Wire([SplineInterpolation(pnts, periodic=True)]))
cavity.faces.name = "cavity"
cavity.faces.maxh = 0.08

c = (0.3,0)    # center of the computational domain
air = Circle(c, 3).Face() - cavity
air.faces.name = "air"
pmlregion = Circle(c, 4).Face() - Circle(c, 3).Face()
pmlregion.faces.name = "pml"
pmlregion.edges.Max(X).name = "outer"

mesh = Mesh(OCCGeometry(Glue([cavity, air, pmlregion]), dim=2).GenerateMesh(maxh=0.3)).Curve(5)
mesh.SetPML(pml.Radial(rad=3, alpha=1j, origin=c), "pml")
Draw (mesh);

In [ ]:
n = mesh.MaterialCF({"cavity" : 3.3}, default=1)
fes = H1(mesh, order=5, complex=True, dirichlet="outer")
u, v = fes.TnT()
a = BilinearForm(1/n**2*grad(u)*grad(v)*dx).Assemble()
m = BilinearForm(u*v*dx).Assemble()

modes = GridFunction(fes, multidim=80)
with TaskManager():
    lam = np.array(ArnoldiSolver(a.mat, m.mat, fes.FreeDofs(), list(modes.vecs), shift=8**2))

Arnoldi returns 80 Ritz values around the shift $k = 8$, the outer ones are not converged. We keep the eigenpairs with a small residual $\| A u - \lambda M u \|$:

In [ ]:
proj = Projector(fes.FreeDofs(), True)
r, w = a.mat.CreateVector(), a.mat.CreateVector()
def Residual(lam, vec):
    r.data = proj * (a.mat*vec - lam*m.mat*vec)
    w.data = m.mat*vec
    return Norm(r) / (abs(lam)*Norm(w))

conv = [i for i in range(len(lam)) if Residual(lam[i], modes.vecs[i]) < 1e-8]
print (len(conv), "of", len(lam), "eigenpairs are converged")
kres = np.sqrt(lam[conv])
Q = kres.real / (-2*kres.imag)

Resonances in the complex $k$-plane: long-lived modes close to the real axis, and a band of leaky modes below:

In [ ]:
plt.figure(figsize=(8,3))
plt.plot(kres.real, kres.imag, "o")
plt.xlabel("Re k"); plt.ylabel("Im k"); plt.grid(True)
plt.show()

order = np.argsort(-Q)
for i in order[:6]:
    print (f"k = {kres[i]:.5f},   Q = {Q[i]:8.1f}")

The mode with the highest $Q$: a whispering gallery pattern inside the cavity, and outside the light leaves to the right.
We look at the intensity with a logarithmic scale:

In [ ]:
def Mode(i):
    gf = GridFunction(fes)
    gf.vec.data = modes.vecs[conv[i]]
    gf.vec.data *= 1/sqrt(Integrate(Norm(gf)**2, mesh, definedon=mesh.Materials("cavity")))
    return gf

mode = Mode(order[0])
intensity = mesh.MaterialCF({"pml" : -7}, default=log(Norm(mode)**2+1e-12)/log(10))
Draw (intensity, mesh, order=4, min=-7, max=0, settings={"Objects" : {"Wireframe" : False}});

Emission direction: $|u|^2$ on the circle of radius 2.8 around the center, for the six modes with the highest $Q$. They come in pairs, even and odd in $y$, and all emit into the same direction:

In [ ]:
angles = np.linspace(0, 2*pi, 361)
circle = mesh(c[0]+2.8*np.cos(angles), c[1]+2.8*np.sin(angles))
plt.figure(figsize=(4,4))
for i in order[:6]:
    emission = abs(Mode(i)(circle).flatten())**2
    plt.polar(angles, emission / max(emission))
plt.gca().set_yticklabels([])
plt.title("emission pattern")
plt.show()

## More on waves in NGSolve

* 3D Maxwell with $H(\operatorname{curl})$ elements: waveguides, photonic structures, resonators, Bloch modes with periodic boundary conditions
* exceptional points: a resonance pair under a changing pump, in the [next section](exceptional.ipynb)
* time domain: explicit DG methods on GPUs, matrix-free
* nonlinear eigenvalue problems from dispersive materials or Hardy space infinite elements
* boundary element methods for unbounded exteriors (`ngsolve.bem`)